# Business Forecasting
## Exponential smoothing in Python
**Amir Hosseinipour · Student learning notebook**

Read each explanation, run the next cell, then explain the result in your own words.

**Learning goals:** calculate a simple smoothing update, explain smoothing parameters, fit trend and seasonal models, evaluate forecasts on future observations, and produce a 24-month forecast.

**Google Colab:** upload this `.ipynb` at https://colab.research.google.com/ using **File → Upload notebook**. Select **Runtime → Run all**. No uploaded dataset or API key is required. A CPU runtime is sufficient.

### Route through the lesson
1. Understand level, trend, and seasonality.
2. Calculate simple smoothing by hand and in Python.
3. Fit SES, Holt, additive and multiplicative Holt–Winters, and damped models.
4. Compare 24-month forecasts with benchmarks.
5. Diagnose errors and forecast the next 24 months.
6. Complete the practice questions.

**Data:** the historical AirPassengers series, monthly international airline passengers, 1949–1960, in thousands. The 144 observations are embedded below. This is the standard airline series commonly used for the SASHELP.AIR exercise. Check your SAS copy's dates and values before expecting identical estimates. Initialization and software settings can also change estimates.

**Scope note:** “double exponential smoothing” often means Holt's level-and-trend method, but can also refer to Brown's more restricted method. Here we use Holt explicitly. Seasonal Holt–Winters adds a seasonal component. The notebook labels each model by its components to avoid confusion.

## 1. Set up Python
Colab normally supplies these packages. The first cell installs the versions used for the forecasting and notebook calculations. A notebook cell can use `%pip`; a regular `.py` file cannot. If Colab asks for a restart after installation, restart the session and run all cells again.

In [ ]:
%pip -q install "statsmodels==0.14.6" "numpy<3" "pandas<3" "matplotlib<4" "scipy<2"

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy
import statsmodels
from IPython.display import display
from statsmodels.tsa.holtwinters import SimpleExpSmoothing, Holt, ExponentialSmoothing
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.stats.diagnostic import acorr_ljungbox
plt.rcParams.update({"figure.figsize": (11, 4), "axes.grid": True})
pd.set_option("display.max_columns", 12)
print("Python:", sys.version.split()[0])
print("numpy:", np.__version__, "pandas:", pd.__version__)
print("statsmodels:", statsmodels.__version__, "scipy:", scipy.__version__)

## 2. Load and inspect the real data
Each number is one month, beginning in January 1949. `freq="MS"` means monthly observations dated at the start of the month. A value of 112 means 112,000 passengers. The historical dates matter: the forecast after December 1960 begins in January 1961. These are teaching forecasts made from a historical endpoint, not current airline forecasts.

In [ ]:
values = [
112,118,132,129,121,135,148,148,136,119,104,118,
115,126,141,135,125,149,170,170,158,133,114,140,
145,150,178,163,172,178,199,199,184,162,146,166,
171,180,193,181,183,218,230,242,209,191,172,194,
196,196,236,235,229,243,264,272,237,211,180,201,
204,188,235,227,234,264,302,293,259,229,203,229,
242,233,267,269,270,315,364,347,312,274,237,278,
284,277,317,313,318,374,413,405,355,306,271,306,
315,301,356,348,355,422,465,467,404,347,305,336,
340,318,362,348,363,435,491,505,404,359,310,337,
360,342,406,396,420,472,548,559,463,407,362,405,
417,391,419,461,472,535,622,606,508,461,390,432]
y = pd.Series(values, index=pd.date_range("1949-01-01", periods=144, freq="MS"),
              name="Passengers_thousands", dtype=float)
assert len(y) == 144 and y.notna().all() and (y > 0).all()
display(y.to_frame().head(12))
display(y.describe().to_frame())
y.plot(title="International airline passengers, 1949–1960", ylabel="Passengers (thousands)")
plt.show()

### Read the plot before choosing a model
**Level** is the current baseline. **Trend** is the sustained direction of change. **Seasonality** is the recurring pattern within the year. Here the series rises and the summer peaks repeat. The distance between peaks and troughs also grows as passenger numbers increase. That makes multiplicative seasonality a sensible candidate, but we still evaluate it against other models.

Monthly annual seasonality uses **m = 12**. Quarterly annual seasonality uses **m = 4**. The value is observations per seasonal cycle; it is not automatically 12 for every dataset.

**Pause:** Would a horizontal forecast capture the summer peaks? Why might a trend-only forecast still miss important information?

In [ ]:
season_view = pd.DataFrame({"Passengers": y, "Year": y.index.year, "Month": y.index.month})
season_view.pivot(index="Month", columns="Year", values="Passengers").plot(
    title="Within-year patterns: one line per year", ylabel="Passengers (thousands)")
plt.xticks(range(1, 13)); plt.show()

## 3. Simple exponential smoothing (SES) · slides 2–6
SES is suitable when a series has no clear trend or seasonal pattern. We apply it to the airline series as a deliberately simple comparison.

$$\ell_t=\alpha y_t+(1-\alpha)\ell_{t-1},\qquad \hat y_{t+h|t}=\ell_t.$$

- $y_t$: actual observation at time $t$.
- $\ell_t$: level after observing $y_t$.
- $\alpha$: weight placed on the newest observation, between 0 and 1.
- $h$: forecast horizon, measured in months here.

If the previous level is 100, the new observation is 120, and $\alpha=0.3$, the updated level is $0.3(120)+0.7(100)=106$. Every forecast made from this endpoint equals 106 until another observation arrives.

A high alpha reacts quickly. A low alpha smooths more strongly. At alpha = 1, the next forecast equals the latest observation, which is the naïve method. Expanding the recursion gives weights $\alpha,\alpha(1-\alpha),\alpha(1-\alpha)^2,\ldots$ for recent observations, plus a remaining weight on the initial level.

In [ ]:
alpha = 0.3
previous_level = 100
new_observation = 120
print("Updated level:", alpha * new_observation + (1-alpha) * previous_level)

def manual_ses(series, alpha, initial_level):
    """Forecast before observing each value; then update the level."""
    level = float(initial_level)
    rows = []
    for date, actual in series.items():
        forecast = level
        error = actual - forecast
        level = alpha * actual + (1-alpha) * level
        rows.append([date, actual, forecast, error, level])
    return pd.DataFrame(rows, columns=["Month", "Actual", "Forecast", "Error", "Updated_level"]).set_index("Month")

# A simple teaching initialization, not an optimized initial state.
worked = manual_ses(y.iloc[:8], alpha=0.3, initial_level=y.iloc[0])
display(worked.round(3))

### Estimating alpha and the initial level
An error is actual minus forecast: $e_t=y_t-\hat y_{t|t-1}$. A positive error means the forecast was too low.

$$SSE=\sum_{t=1}^{T}e_t^2.$$

Software searches for parameters and initial states that reduce the training SSE. This optimization is numerical, so it can depend on starting values and constraints. A small training SSE does not guarantee accurate forecasts of future months. The manual example fixes the initial level for transparency; the main models estimate it using the training sample.

The next section splits the data **before** fitting. That prevents future observations from influencing training estimates.

In [ ]:
H = 24
train, test = y.iloc[:-H], y.iloc[-H:]
assert train.index[-1] < test.index[0]
print("Train:", train.index[0].date(), "to", train.index[-1].date(), "n =", len(train))
print("Test: ", test.index[0].date(), "to", test.index[-1].date(), "n =", len(test))
for a in [0.1, 0.5, 0.9, 1.0]:
    demo = SimpleExpSmoothing(train, initialization_method="known", initial_level=train.iloc[0]).fit(
        smoothing_level=a, optimized=False)
    plt.plot(demo.fittedvalues.index, demo.fittedvalues, label=f"alpha={a}")
plt.plot(train, color="black", alpha=0.4, label="Actual")
plt.title("One-step fitted forecasts with different alpha values")
plt.ylabel("Passengers (thousands)"); plt.legend(); plt.show()

## 4. Holt's linear trend · slides 8–10
Holt updates both the baseline and the slope:

$$\ell_t=\alpha y_t+(1-\alpha)(\ell_{t-1}+b_{t-1})$$
$$b_t=\beta^*(\ell_t-\ell_{t-1})+(1-\beta^*)b_{t-1}$$
$$\hat y_{t+h|t}=\ell_t+h b_t.$$

The slope $b_t$ is a change per month. If the final level is 300 and the slope is 4, forecasts for the next 3 months are 304, 308, and 312. `smoothing_trend` is the trend smoothing parameter. It controls how quickly the estimated slope changes; it is different from the slope itself.

Holt's forecast can rise or fall, but without a seasonal component it cannot reproduce repeating summer peaks. The numerical parameters printed in a slide belong to that slide's example and are not universal settings.

## 5. Holt–Winters seasonality · slides 11–13
### Additive seasonality
Use additive seasonality when seasonal changes are approximately constant in units.

$$\ell_t=\alpha(y_t-s_{t-m})+(1-\alpha)(\ell_{t-1}+b_{t-1})$$
$$b_t=\beta^*(\ell_t-\ell_{t-1})+(1-\beta^*)b_{t-1}$$
$$s_t=\gamma(y_t-\ell_{t-1}-b_{t-1})+(1-\gamma)s_{t-m}$$
$$\hat y_{t+h|t}=\ell_t+h b_t+s_{\text{matching season}}.$$

A seasonal component of +40 means about 40,000 passengers above the trend baseline. Normalized additive seasonal components sum to approximately zero over a cycle.

### Multiplicative seasonality
Use multiplicative seasonality when the seasonal size changes in proportion to the level.

$$\ell_t=\alpha\frac{y_t}{s_{t-m}}+(1-\alpha)(\ell_{t-1}+b_{t-1})$$
$$b_t=\beta^*(\ell_t-\ell_{t-1})+(1-\beta^*)b_{t-1}$$
$$s_t=\gamma\frac{y_t}{\ell_{t-1}+b_{t-1}}+(1-\gamma)s_{t-m}$$
$$\hat y_{t+h|t}=(\ell_t+h b_t)s_{\text{matching season}}.$$

A factor of 1.20 means 20% above baseline. Normalized multiplicative factors average approximately 1. Multiplicative models require strictly positive observations. Our models retain an **additive trend** even when seasonality is multiplicative.

**Implementation note:** these equations use the seasonal-update convention shown in the lecture/FPP. The `statsmodels.tsa.holtwinters` implementation updates seasonality using the newly updated level. Parameter conventions can therefore differ. We use the library's implementation consistently for fitting and forecasting; do not expect an identically numbered gamma to reproduce every textbook recursion.

## 6. Damped trend · slide 14
Straight-line growth can become unrealistic at a long horizon. Damping gradually reduces the trend contribution:

$$\hat y_{t+h|t}=\ell_t+(\phi+\phi^2+\cdots+\phi^h)b_t$$

with seasonal adjustment added or multiplied when seasonality is included. For $0<\phi<1$, each extra step adds less trend. At $\phi=1$ we recover an undamped trend. Damping approaches a limiting trend contribution rather than becoming exactly flat at a particular finite date.

For damped Holt, replace the previous trend contribution by $\phi b_{t-1}$ in the level update, and use $\phi b_{t-1}$ as the carried-forward trend in the trend update. The fitted damping parameter is printed below.

In [ ]:
horizons = np.arange(1, 25)
plt.plot(horizons, 300 + 4*horizons, label="Undamped: level 300, slope 4")
plt.plot(horizons, 300 + 4*np.cumsum(0.9**horizons), label="Damped: phi 0.9")
plt.xlabel("Months ahead"); plt.ylabel("Illustrative forecast")
plt.title("How damping changes the trend contribution"); plt.legend(); plt.show()

## 7. Fit the candidate models
All models see only January 1949–December 1958. The test period is January 1959–December 1960. We forecast those 24 months from one fixed endpoint, without updating the models with test observations.

| Model | Trend | Seasonality |
|---|---|---|
| SES | None | None |
| Holt | Additive | None |
| Damped Holt | Damped additive | None |
| HW additive | Additive | Additive, m=12 |
| HW multiplicative | Additive | Multiplicative, m=12 |
| Damped HW multiplicative | Damped additive | Multiplicative, m=12 |

`initialization_method="estimated"` estimates initial states. `optimized=True` estimates smoothing parameters. `use_brute=True` searches for useful starting values. It does not prove a global optimum. No log transformation or bias correction is applied.

In [ ]:
def fit_models(series):
    """Fit the same predeclared candidates to any training series."""
    return {
        "SES": SimpleExpSmoothing(series, initialization_method="estimated").fit(optimized=True, use_brute=True),
        "Holt": Holt(series, initialization_method="estimated").fit(optimized=True, use_brute=True),
        "Damped Holt": Holt(series, damped_trend=True, initialization_method="estimated").fit(optimized=True, use_brute=True),
        "HW additive": ExponentialSmoothing(series, trend="add", seasonal="add", seasonal_periods=12,
            initialization_method="estimated").fit(optimized=True, use_brute=True),
        "HW multiplicative": ExponentialSmoothing(series, trend="add", seasonal="mul", seasonal_periods=12,
            initialization_method="estimated").fit(optimized=True, use_brute=True),
        "Damped HW multiplicative": ExponentialSmoothing(series, trend="add", damped_trend=True,
            seasonal="mul", seasonal_periods=12, initialization_method="estimated").fit(optimized=True, use_brute=True)
    }
fits = fit_models(train)
parameter_names = ["smoothing_level", "smoothing_trend", "smoothing_seasonal",
                   "damping_trend", "initial_level", "initial_trend"]
parameters = pd.DataFrame({name: {p: fit.params.get(p, np.nan) for p in parameter_names}
                           for name, fit in fits.items()}).T
display(parameters.round(4))
for name, fit in fits.items():
    result = getattr(fit, "mle_retvals", None)
    print(name, "optimizer success:", result.get("success", "not reported") if result is not None else "not reported")

### Interpret the parameter table
A blank/NaN parameter means that component is absent. A value close to 1 means strong updating of that component; a value close to 0 means very slow updating. A small seasonal smoothing parameter can still accompany a strong seasonal pattern: the pattern may simply be stable over time.

The initial level and trend are estimated starting states, not the final level and slope. Parameters near boundaries deserve inspection, but do not by themselves prove that a model is wrong. If optimization reports failure or prints a convergence warning, investigate before relying on that fit.

In [ ]:
state_fit = fits["HW multiplicative"]
states = pd.DataFrame({"Actual": train, "Level": state_fit.level,
                       "Trend_per_month": state_fit.trend, "Seasonal_factor": state_fit.season})
display(states.tail(12).round(3))
fig, axes = plt.subplots(3, 1, figsize=(11, 8), sharex=True)
for ax, column in zip(axes, ["Level", "Trend_per_month", "Seasonal_factor"]):
    states[column].plot(ax=ax, title=column)
plt.tight_layout(); plt.show()

## 8. Evaluate forecast errors · slides 16–17
Two benchmarks make the comparison meaningful. **Naïve** repeats the final training value. **Seasonal naïve** repeats the last observed year, month for month, including at horizons beyond 12 months.

For the held-out months, let $e_t=y_t-\hat y_t$:

$$MAE=\frac{1}{n}\sum|e_t|,\quad RMSE=\sqrt{\frac{1}{n}\sum e_t^2},\quad MAPE=\frac{100}{n}\sum\left|\frac{e_t}{y_t}\right|.$$

MAE and RMSE use thousands of passengers. RMSE penalizes large misses more strongly. MAPE is a percentage and becomes unsuitable with zero or near-zero actuals. All airline values are positive.

We also calculate seasonal **MASE**: MAE divided by the average absolute 12-month difference in the **training** sample. MASE below 1 means the forecast MAE is below that training scaling quantity. It does not necessarily mean the model beats seasonal naïve on this particular test period; compare the test rows directly.

Training SSE measures fitted one-step errors. Test RMSE measures genuine 1-to-24-step forecasts from the cutoff. They answer different questions.

In [ ]:
predictions = pd.DataFrame({name: fit.forecast(H) for name, fit in fits.items()}, index=test.index)
predictions["Naive"] = train.iloc[-1]
predictions["Seasonal naive"] = np.tile(train.iloc[-12:].to_numpy(), int(np.ceil(H/12)))[:H]
scale = train.diff(12).abs().dropna().mean()
def accuracy(actual, forecast):
    error = np.asarray(actual) - np.asarray(forecast)
    return {"MAE": np.mean(np.abs(error)), "RMSE": np.sqrt(np.mean(error**2)),
            "MAPE_percent": 100*np.mean(np.abs(error/np.asarray(actual))),
            "MASE": np.mean(np.abs(error))/scale, "Mean_error": np.mean(error)}
scores = pd.DataFrame({name: accuracy(test, predictions[name]) for name in predictions}).T.sort_values("RMSE")
training_sse = pd.Series({name: fit.sse for name, fit in fits.items()}, name="Training_SSE")
display(training_sse.sort_values().round(2).to_frame())
display(scores.round(3))
print("Lowest test RMSE among these candidates:", scores.index[0])
print("Positive mean error means underforecasting on average.")

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(14, 11), sharex=True)
for ax, name in zip(axes.ravel(), fits):
    ax.plot(train.iloc[-36:], label="Training")
    ax.plot(test, color="black", label="Test actual")
    ax.plot(predictions[name], label=name)
    ax.plot(predictions["Seasonal naive"], linestyle=":", label="Seasonal naive")
    ax.axvline(test.index[0], color="gray", linestyle="--")
    ax.set_title(name); ax.set_ylabel("Passengers (thousands)"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()
display(pd.concat([test.rename("Actual"), predictions], axis=1).round(2))

### Explain what you found
SES gives a flat forecast because its forecast equation contains only a level. Holt adds a slope but still misses the recurring summer pattern. Seasonal models can reproduce that pattern. Multiplicative seasonality is motivated by the increasing size of the swings, but the error table determines whether it performs better over this particular evaluation period.

Look for both timing and size: Does the model place peaks in the correct months? Does it predict peaks that are too low? Does damping help at longer horizons?

**Selection caution:** choosing a winner using 1959–1960 makes this period a validation set for model selection. Its winning error is useful for classroom comparison but is not a fresh, unbiased estimate of the selected model's future performance. A separate untouched test period or rolling-origin evaluation is preferable for a formal forecasting study. Rankings may change with the cutoff and horizon.

**Results in the included run:** multiplicative Holt–Winters has RMSE about 32.49 thousand passengers and MAPE about 6.39%, versus seasonal naïve RMSE about 77.00 thousand. Its mean error is about +28.66 thousand, so it still tends to forecast too low. Additive Holt–Winters has RMSE about 35.76 thousand. The multiplicative model performs better here but leaves meaningful systematic error. These numbers come from the included execution; use your own regenerated table if software settings change.

## 9. Residual diagnostics
A training residual compares an actual value with its fitted one-step forecast. A test forecast error compares an unseen observation with the forecast made at the training cutoff.

The residual time plot checks for persistent bias or changing variability. The ACF checks whether residuals still contain serial dependence. A peak at lag 12 can indicate remaining annual structure. The Ljung–Box test checks groups of residual autocorrelations. A small p-value gives evidence against the no-autocorrelation null under the test assumptions; a large p-value does not prove a correct model.

We use this test as a classroom screen with `model_df=0`. The displayed p-values are approximate and do not adjust for all estimated smoothing parameters and initial states. Do not choose a forecasting model solely from this test.

In [ ]:
# Diagnose a predeclared seasonal model rather than choosing it from residual p-values.
resid = pd.Series(fits["HW multiplicative"].resid, index=train.index)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
resid.plot(ax=axes[0], title="Training residuals: HW multiplicative")
axes[0].axhline(0, color="black", linewidth=1)
plot_acf(resid, lags=24, ax=axes[1], zero=False)
plt.tight_layout(); plt.show()
display(acorr_ljungbox(resid, lags=[12, 24], model_df=0).round(4))
print("Average training residual:", round(resid.mean(), 3))

## 10. Refit and forecast 24 future months
After the classroom comparison, we refit the smoothing model with the lowest validation RMSE using all observations through December 1960. We select among smoothing candidates here; the benchmark rows remain important checks. If a benchmark is better, report that explicitly.

The forecasts cover January 1961–December 1962. These are **point forecasts**. They do not provide a range for uncertainty. Wider uncertainty is usually expected at longer horizons. Prediction intervals require an explicit stochastic model or a justified simulation procedure; the notebook does not invent intervals from RMSE.

In [ ]:
selected_name = scores.loc[list(fits)].sort_values("RMSE").index[0]
all_fits = fit_models(y)
final_fit = all_fits[selected_name]
future = final_fit.forecast(24).rename("Forecast_thousands")
print("Selected smoothing model:", selected_name)
print("Overall validation winner including benchmarks:", scores.index[0])
display(future.round(2).to_frame())
plt.plot(y, label="Observed through 1960")
plt.plot(future, label=f"24-month forecast: {selected_name}")
plt.title("Historical forecast made at December 1960")
plt.ylabel("Passengers (thousands)"); plt.legend(); plt.show()
# Convert units only when reporting individual passengers.
print("First forecast in passengers:", round(future.iloc[0] * 1000))

## 11. Save tables and source data
The next cell writes CSV files into the current Colab session. Session files are temporary. Use the file panel to download them, or run the optional download cell. Re-running overwrites files with the same names.

In [ ]:
from pathlib import Path
output_dir = Path("week7_outputs")
output_dir.mkdir(exist_ok=True)
y.to_csv(output_dir / "air_passengers.csv", index_label="Month")
scores.to_csv(output_dir / "validation_accuracy.csv", index_label="Model")
parameters.to_csv(output_dir / "training_parameters.csv", index_label="Model")
pd.concat([test.rename("Actual"), predictions], axis=1).to_csv(output_dir / "validation_forecasts.csv", index_label="Month")
future.to_csv(output_dir / "future_24_month_forecast.csv", index_label="Month")
print("Saved:", [p.name for p in sorted(output_dir.glob("*.csv"))])

In [ ]:
# Optional: change to True to download all CSVs when running in Colab.
DOWNLOAD_RESULTS = False
if DOWNLOAD_RESULTS:
    import shutil
    archive = shutil.make_archive("Week7_Results", "zip", output_dir)
    try:
        from google.colab import files
        files.download(archive)
    except ImportError:
        print("Outside Colab: open", archive, "from your working directory.")

## 12. Student practice
Write your answers in new text cells. Keep numerical results alongside the explanation.

1. With a previous level of 200, an actual value of 230, and alpha = 0.2, calculate the new SES level and the 1-month and 6-month forecasts.
2. Explain why alpha = 1 gives the naïve forecast. What happens when alpha = 0 with a fixed initial level?
3. If Holt's final level is 400 and its final slope is 3, calculate forecasts for horizons 1, 6, and 24. Explain the units of the slope.
4. A July seasonal component is +60 in one model and 1.15 in another. Explain the difference using a trend baseline of 400.
5. Compare SES, Holt, and both undamped seasonal models. Report validation MAE, RMSE, and MAPE. Explain why the forecasts differ.
6. Does the model with the lowest training SSE also have the lowest validation RMSE? Use the actual tables.
7. Compare the best smoothing model with seasonal naïve. Is the improvement large enough to matter for staffing decisions? State a practical cost or constraint that the error measures leave out.
8. Change the holdout horizon to 12 by editing `H = 24` in Section 3, then rerun from that cell through Section 9. Does the ranking change? Restore 24 afterward. The deployment forecast in Section 10 intentionally remains 24 months.
9. Explain why randomly shuffling these observations before splitting would give an inappropriate evaluation.
10. Write a 150-word recommendation naming a model, citing an error measure and a plot, and stating one limitation.

### Check your calculations after trying
<details><summary>Answers to Questions 1–4</summary>

1. New level = 206. SES gives 206 at both horizons from this endpoint.
2. Alpha = 1 replaces the level with the newest actual. Alpha = 0 keeps the supplied initial level unchanged.
3. Forecasts = 403, 418, and 472, all in thousands of passengers. The slope is 3,000 passengers per month.
4. Additive July forecast = 460. Multiplicative July forecast = 460 as well at this baseline, but the second model uses a 15% uplift. At a baseline of 800 they give 860 and 920 respectively.

</details>

### Submission checklist
Include your notebook with visible outputs, the comparison table, the 24-month future forecast, and your recommendation. Explain results in words rather than pasting tables alone. Keep the chronological split and identify the units.

## Troubleshooting
- **NameError:** run the earlier cells in order, or use Runtime → Run all.
- **Multiplicative model error:** inspect zero/negative values and missing data. Do not add a constant without explaining how it changes the problem.
- **Convergence warning:** inspect optimizer status, data scale, and initialization. Do not silently hide the warning.
- **Slightly different results:** check package versions and settings. Numerical optimization can produce small differences.
- **Date problems:** verify one observation per month in order, with no duplicates or gaps.
- **Monthly forecasts without seasonality:** check `seasonal_periods=12` and whether `seasonal` was supplied.

## References and further reading
- Hyndman & Athanasopoulos, *Forecasting: Principles and Practice*, Chapter 8: https://otexts.com/fpp3/exponential-smoothing.html. See simple, trend, and seasonal methods and model selection.
- Statsmodels Holt–Winters API: https://www.statsmodels.org/stable/generated/statsmodels.tsa.holtwinters.ExponentialSmoothing.html
- Statsmodels worked examples: https://www.statsmodels.org/stable/examples/notebooks/generated/exponential_smoothing.html
- AirPassengers data documentation in R: https://stat.ethz.ch/R-manual/R-devel/library/datasets/html/AirPassengers.html. Source credited there: Box & Jenkins (1976).

**Method reminder:** the nine basic trend/seasonal method combinations in a smoothing taxonomy describe component choices. ETS additionally specifies an error model. Multiplicative seasonality does not automatically imply multiplicative errors. This lesson fits classical Holt–Winters smoothing models and compares forecast accuracy.